In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")

In [3]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"]= TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(model="models/embedding-001")

d:\LangGraph-end-to-end\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [6]:
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [35]:
from langchain_groq import ChatGroq
llm_groq=ChatGroq(model="qwen/qwen3-32b")

# Simple Ai Assistant

In [8]:
result=llm.invoke("Introduce yourself in 3 sentences.")
print(result)

content="Hello! I am a large language model, trained by Google. I'm here to assist you by providing information, generating creative content, and helping with various tasks. Feel free to ask me anything or tell me what you'd like to explore!" additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cfcca-5060-7ba2-a1c1-81c866eaec48-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 167, 'total_tokens': 175, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 116}}


In [9]:
print(llm_groq.invoke("Introduce yourself in 3 sentences."))

content="<think>\nOkay, the user wants me to introduce myself in three sentences. Let me start by recalling my main functions. I'm Qwen, a large language model developed by Alibaba Cloud's Tongyi Lab. I should mention my purpose, which is to assist with information and tasks. Next, I need to highlight my capabilities, like answering questions, creating text, and supporting multiple languages. Also, maybe mention some specific features like coding. Finally, I should express my goal to be helpful and user-friendly. Let me structure each sentence to cover these points concisely. Make sure it's clear and not too technical. Let me check if three sentences are enough. First sentence: who I am and my purpose. Second: what I can do. Third: my goal and invitation to help. Yeah, that should work. Let me put it all together in English now.\n</think>\n\nHello! I am Qwen, a large language model developed by Alibaba Cloud's Tongyi Lab, designed to assist with a wide range of tasks, from answering qu

Adding a Memory to the AI Assistant

In [10]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.messages import AIMessage, HumanMessage

In [11]:
store={}

In [15]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [12]:
config={'configurable':{'session_id': 'firstchat'}}

In [16]:
model_with_memory=RunnableWithMessageHistory(llm_groq, get_session_history)

In [18]:
model_with_memory.invoke(("Hi I'm Harshanth"),config=config).content

'<think>\nOkay, the user started by saying, "Hi I\'m Harshanth" and I responded with a greeting. Now they\'re saying "Hi I\'m Harshanth" again. Maybe they\'re testing if I remembered their name or just being friendly again. Let me check the previous messages.\n\nIn the first message, they introduced themselves, so I acknowledged them. Now they\'re repeating the same line. Could be a typo, but likely they just want to start a conversation. Since I already greeted them, maybe they need help with something else. I should ask how I can assist them today. Keep it friendly and open-ended. Make sure to use their name to personalize the response. Let me make sure the tone is welcoming and not repetitive. Alright, that should cover it.\n</think>\n\nHello again, Harshanth! 😊 How can I help you today? Whether you have a question, need advice, or just want to chat, I\'m here for you!'

In [19]:
print(model_with_memory.invoke(("what is my name"),config=config).content)

<think>
Okay, the user just asked, "what is my name." Let me check the history. The first message was the user introducing themselves as Harshanth. The assistant responded with a friendly greeting. Then the user said "Hi I'm Harshanth" again, and the assistant acknowledged that. Now, the user is asking for their name.

Hmm, the user might be testing if the assistant remembers their name. Since the user provided their name in the first interaction, the assistant should recall that. But maybe the user is confused or wants confirmation. The previous interactions were straightforward, so the assistant should confirm the name again.

Wait, the user's name is Harshanth. The assistant's last response was a greeting, so the user might be checking if the name is stored correctly. The assistant needs to clearly state that the name is Harshanth. Also, maybe the user is having an issue with the system not remembering their name, but since the conversation is in the same thread, it should remember.

In [21]:
print(store['firstchat'])

Human: Hi I'm Harshanth
AI: <think>
Okay, the user has introduced themselves as Harshanth. I need to respond in a friendly and welcoming manner. Let me make sure to acknowledge their introduction and offer assistance.

First, I should greet them and say it's nice to meet them. Then, ask how I can assist them today. Keep it simple and open-ended so they feel comfortable sharing what they need help with.

I should check if there's any specific context or previous interactions, but since this is the start, just a general offer to help. Make sure the tone is positive and approachable.
</think>

Hi Harshanth! Nice to meet you. How can I assist you today? 😊
Human: Hi I'm Harshanth
AI: <think>
Okay, the user started by saying, "Hi I'm Harshanth" and I responded with a greeting. Now they're saying "Hi I'm Harshanth" again. Maybe they're testing if I remembered their name or just being friendly again. Let me check the previous messages.

In the first message, they introduced themselves, so I ac

# RAG with LCEL(Langchain Expression Chain Language)

In [24]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Reading the text files from source directory
loader=DirectoryLoader('../data', glob='./*.txt*', loader_cls=TextLoader)
docs=loader.load()

In [ ]:
# creating Chunks using RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=100, 
    chunk_overlap=20,
    length_function=len
    )

In [ ]:
new_docs=text_splitter.split_documents(documents=docs)
doc_strings=[doc.page_content for doc in new_docs]

In [26]:
# BGE Embedding
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}  # set True to compute cosine similarity

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

C:\Users\USER\AppData\Local\Temp\ipykernel_17568\1564669262.py:8: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
d:\LangGraph-end-to-end\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https

In [ ]:
# creating retriever using Chroma vector store
db=Chroma.from_documents(new_docs, embeddings)
retriever=db.as_retriever(search_kwargs={"k":4})

In [ ]:
template="""
You are a RAG assistant for question answering.
Answer the question based only on the follwing content:
context_retrieved:{context}
question:{question}
if the context does not contain the answer, say you don't know.
if the question is not clear, ask for clarification or the question and retrived context is both different then say you don't know.
"""


In [ ]:
prompt=PromptTemplate.from_template(template)

In [ ]:
retrival_chain=(
    RunnableParallel({'context':retriever, "question":RunnablePassthrough()})
    | prompt
    | llm_groq
    | StrOutputParser()
)

In [ ]:
question="What is llama? can you highlight the answer in the context if you know?"

In [ ]:
print(retrival_chain.invoke(question))

# Start with Tools and Agents

In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [2]:
api_wrapper=WikipediaAPIWrapper()

In [3]:
Wikipedia_tool=WikipediaQueryRun(api_wrapper=api_wrapper)

In [4]:
Wikipedia_tool

WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\LangGraph-end-to-end\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000))

In [5]:
Wikipedia_tool.name

'wikipedia'

In [6]:
Wikipedia_tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [8]:
Wikipedia_tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [11]:
print(Wikipedia_tool.run({'query':'latest news on AI'}))

Page: Mistral AI
Summary: Mistral AI SAS (French: [mistʁal]) is a French artificial intelligence (AI) company, headquartered in Paris. Founded in 2023, it has open-weight large language models (LLMs), with both open-source and proprietary AI models. As of 2025 the company has a valuation of more than US$14 billion.

Page: Meta AI
Summary: Meta AI is a research division of Meta (formerly Facebook) that develops artificial intelligence and augmented reality technologies.



Page: AI slop
Summary: AI slop (also known simply as slop) is digital content made with generative artificial intelligence that is perceived as lacking in effort, quality, or meaning, and produced in high volume as clickbait to gain advantage in the attention economy, or earn money. It is a form of synthetic media usually linked to the monetization in the creator economy of social media and online advertising. Coined in the 2020s, the term has a pejorative connotation similar to spam. "Slop" was selected as the 2025 W

In [14]:
# youtube search
from langchain_community.tools import YouTubeSearchTool

In [15]:
youtube_tool=YouTubeSearchTool()

In [16]:
youtube_tool.args

{'query': {'title': 'Query', 'type': 'string'}}

In [17]:
youtube_tool.description

'search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is the maximum number of video results to return aka num_results. the second part is optional'

In [ ]:
!pip install youtube_search

In [21]:
print(youtube_tool.run('latest news on AI'))

['https://www.youtube.com/watch?v=W31ro8YT7jc&pp=ygURbGF0ZXN0IG5ld3Mgb24gQUk%3D', 'https://www.youtube.com/watch?v=qWzo3ws0uWU&pp=ygURbGF0ZXN0IG5ld3Mgb24gQUk%3D']


In [22]:
from langchain_community.tools import TavilyAnswer

In [23]:
websearch_tool_tavily=TavilyAnswer()

C:\Users\USER\AppData\Local\Temp\ipykernel_9964\409000541.py:1: LangChainDeprecationWarning: The class `TavilyAnswer` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  websearch_tool_tavily=TavilyAnswer()


In [24]:
websearch_tool_tavily.args

{'query': {'description': 'search query to look up',
  'title': 'Query',
  'type': 'string'}}

In [ ]:
websearch_tool_tavily.run('latest news on AI')

"Meta unveiled new custom AI chips for its data centers, while Nvidia resumed production of AI chips for China. Microsoft is reportedly considering legal action over Amazon's $50 billion cloud deal with OpenAI."

In [28]:
!pip show langchain

Name: langchain
Version: 1.2.12
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: D:\LangGraph-end-to-end\.venv\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [29]:
import langchain.agents as agents
print(dir(agents))

['AgentState', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'create_agent', 'factory', 'middleware', 'structured_output']


In [31]:
from langchain.agents import create_agent

In [32]:
tools=[Wikipedia_tool, youtube_tool, websearch_tool_tavily]

In [ ]:
agent=create_agent(
    model=llm_groq,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when needed “If one tool is insufficient, try another relevant tool before final answer"
)

In [37]:
result=agent.invoke({
    'messages':[
        {'role':'user','content':'Give me latest Ai news and summarize in 5 points'}
    ]
})

In [43]:
result

{'messages': [HumanMessage(content='Give me latest Ai news and summarize in 5 points', additional_kwargs={}, response_metadata={}, id='56b19dff-2e0f-4987-a209-78abab15cceb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the latest AI news summarized in 5 points. Let me see which tool I should use here. The available tools are Wikipedia, YouTube search, and Tavily Answer. \n\nFirst, Wikipedia is good for general knowledge and factual information, but the user is asking for the latest news, which might not be up-to-date on Wikipedia. YouTube search could help find videos, but the user wants a summary of news articles, not videos. \n\nTavily Answer is described as a search engine optimized for current events. That sounds perfect because the user needs the latest information. The Tavily function takes a search query and returns the answer directly. \n\nSo I should use Tavily Answer with the query "latest AI news" to get the most recent updat

In [54]:
print(result['messages'][-1].content)

Here’s a summary of the latest AI news in 5 points:  

1. **Microsoft vs. Amazon-OpenAI**: Microsoft is considering legal action over Amazon’s cloud partnership with OpenAI, highlighting tensions in AI industry alliances.  
2. **Meta’s AI Chips**: Meta announced new AI chips to enhance its AI hardware capabilities, aiming to compete in the AI infrastructure race.  
3. **Major AI Events**: Upcoming conferences like **Ai4** and **NVIDIA GTC** are set to showcase cutting-edge AI innovations and strategies.  
4. **Business Growth**: AI adoption is accelerating across sectors (e.g., healthcare, finance, retail), driving efficiency, automation, and new revenue streams.  
5. **Tech Giants’ Rivalry**: The competition between companies like Microsoft, Meta, and OpenAI underscores the race for AI dominance in cloud, hardware, and software.  

Let me know if you’d like deeper insights on any of these!
